In [1]:
# detailed comments will be added soon ...

import os
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, DefaultDataCollator

from sklearn.model_selection import train_test_split

2024-06-12 10:42:13.559080: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-12 10:42:13.559177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-12 10:42:13.702937: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
train = pd.read_csv('/kaggle/input/chaii-hindi-and-tamil-question-answering/train.csv')
test = pd.read_csv('/kaggle/input/chaii-hindi-and-tamil-question-answering/test.csv')

In [3]:
dataset = Dataset.from_pandas(train)

In [4]:
model_name = 'FacebookAI/xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [5]:
dataset

Dataset({
    features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language'],
    num_rows: 1114
})

In [6]:
# this function is for tokenizing 'context' and 'question' and add answer position into input data

def batch_tokenizer(batch):
    # tokenize input text
    tokenized_inputs = tokenizer(
        batch["context"], batch['question'],
        max_length=512,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_token_type_ids=True
    )
    
    # get answer span
    answer_start = []
    answer_end = []
    
    for i, context in enumerate(batch['context']):
        answer_start.append(batch['answer_start'][i])
        answer_end.append(batch['answer_start'][i] + len(batch['answer_text'][i]))
        
    tokenized_inputs['answer_start'] = answer_start
    tokenized_inputs['answer_end'] = answer_end
    
    tokenized_inputs["input_ids"] = torch.tensor(tokenized_inputs["input_ids"], dtype=torch.long)
    tokenized_inputs["attention_mask"] = torch.tensor(tokenized_inputs["attention_mask"], dtype=torch.long)
    
    return tokenized_inputs

In [7]:
tokenized_dataset = dataset.map(batch_tokenizer, batched=True)

Map:   0%|          | 0/1114 [00:00<?, ? examples/s]

In [8]:
tokenized_dataset

Dataset({
    features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end'],
    num_rows: 1114
})

In [9]:
# after tokenizing, [CLS] and [SEP] will be add to the text, so we need to adjust the answer span.

def adjust_answer_span(example):
    # set [CLS] and [SEP] as default start & end position
    start_position = example["input_ids"].index(tokenizer.cls_token_id)
    end_position = example["input_ids"].index(tokenizer.sep_token_id)
    
    start = False
    end = False
    
    for i, (offset_start, offset_end) in enumerate(example['offset_mapping']):
        if not start and offset_start == example['answer_start']:
            start_position = i
            start = True
        if not end and offset_end == example['answer_end']:
            end_position = i
            end = True
        if start and end:
            break
    
    if not start or not end:
        start_position = -1
        end_position = -1
    
    example['start_positions'] = start_position
    example['end_positions'] = end_position
    
    return example
    

In [10]:
dataset = tokenized_dataset.map(adjust_answer_span, batched=False)

Map:   0%|          | 0/1114 [00:00<?, ? examples/s]

In [11]:
train_indices, validation_indices = train_test_split(range(len(dataset)), test_size=0.2, random_state=42)

dataset = DatasetDict(
    {
        'train' : dataset.select(train_indices), 
        'validation' : dataset.select(validation_indices)
    }
)

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end', 'start_positions', 'end_positions'],
        num_rows: 891
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end', 'start_positions', 'end_positions'],
        num_rows: 223
    })
})

In [13]:
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
data_collator = DefaultDataCollator()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForQuestionAnswering were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
model

XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias

In [15]:
training_args = TrainingArguments(
    report_to='none',
    output_dir="./output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
)

In [16]:
dataset['train']

Dataset({
    features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end', 'start_positions', 'end_positions'],
    num_rows: 891
})

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator
)


trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=224, training_loss=3.543303898402623, metrics={'train_runtime': 110.1135, 'train_samples_per_second': 16.183, 'train_steps_per_second': 2.034, 'total_flos': 465630820503552.0, 'train_loss': 3.543303898402623, 'epoch': 2.0})

In [18]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end', 'start_positions', 'end_positions'],
        num_rows: 891
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answer_text', 'answer_start', 'language', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'answer_end', 'start_positions', 'end_positions'],
        num_rows: 223
    })
})

In [19]:
output = model(
    input_ids=torch.tensor(dataset['validation'][0]['input_ids']).unsqueeze(0).cuda(),
    token_type_ids=torch.tensor(dataset['validation'][0]['token_type_ids']).unsqueeze(0).cuda(),
    attention_mask=torch.tensor(dataset['validation'][0]['attention_mask']).unsqueeze(0).cuda(),
)

In [20]:
start_logits, end_logits = output.start_logits, output.end_logits

In [21]:
start_logits.argmax(-1), end_logits.argmax(-1)

(tensor([10], device='cuda:0'), tensor([4], device='cuda:0'))

In [22]:
dataset['validation'][0]['start_positions'], dataset['validation'][0]['end_positions']

(29, 31)